### Generative AI- Customer Churn Reports

This notebook implements two types of templates using Gemini API to generate customer churn risk summary and retention recommendation, The generated reports will be tested and evaluated to identify which prompt template produces more suitable output.

**Prompt Templates:

-- T1- Zero-Shot: Generates repots using instructions without examples

-- T2- Few-Shot: Generates repots using instructions with worked examples

Both templates use the same instructions and customer information to support a fair comparision

In [ ]:
from pathlib import Path

template_path=Path("Generative_AI/prompts/zero_shot_template.txt")

zero_shot_template = template_path.read_text( encoding="utf-8")

print("Zero-shot template loaded seccessfully")

Zero-shot template loaded seccessfully


In [ ]:
from pathlib import Path

template_path=Path("Generative_AI/prompts/few_shot_template.txt")

few_shot_template = template_path.read_text( encoding="utf-8")

print("Few-shot template loaded successfully")

Few-shot template loaded successfully


In [ ]:
# prepare sample customer data for testing
customer = {
    "churn_prediction": 1,
    "churn_probability": "87%",
    "recency_days":45,
    "loyalty_score": 25,
    "cart_abandonment_rate": "70%"
}
# fill both prompts with the sample customer data
filled_zero_shot = zero_shot_template.format(**customer)
filled_few_shot = few_shot_template.format(**customer)

In [ ]:
# check the filled zero-shot prompt 
print(filled_zero_shot)

You are a customer retention analyst preparing a concise report for business executives.

Task:
Analyze the customer's churn prediction and available customer features. Provide practical retention recommendations


Customer Information:
- Churn Prediction: 1
- Churn Probability: 87%
- Recency Days: 45
- Loyalty Score: 25
- Cart Abandonment Rate: 70%


Instructions:
1. Summarize the customer's estimated churn risk.
2. Identify relevant observations from the provided customer information.
3. Recommend exactly two actionable retention strategies tailored to the customer.
4. Briefly explain the rationale for each strategy.
5. Use clear, professional, executive-friendly English.


Constraints:
- Use only the information provided.
- Do not invent customer details or business facts.
- Do not assume that observed features cause churn.
- Treat churn probability as a model estimate, not a certainty.
- Avoid unsupported guarantees about outcomes.
- If information is insufficient, state the limita

In [ ]:
# check the filled few-shot prompt 
print(filled_few_shot)


You are a customer retention analyst preparing a concise report for business executives.

Task:
Analyze the customer's churn prediction and available customer features. Provide practical retention recommendations.

Instructions:
1. Summarize the customer's estimated churn risk.
2. Identify relevant observations from the provided customer information.
3. Recommend exactly two actionable retention strategies tailored to the customer.
4. Briefly explain the rationale for each strategy.
5. Use clear, professional, executive-friendly English.

Constraints:
- Use only the provided customer information.
- Do not invent facts about the customer.
- Do not claim that customer features cause churn.
- Treat churn probability as a model estimate, not a guaranteed outcome.
- If information is insufficient, acknowledge it.
- Do not include additional sections.

Required Output Format:

Risk Summary:
[Summarize the predicted churn risk]

Retention Recommendations:
1. [Action and brief rationale]
2. [

In [ ]:
from Generative_AI.api_config_template import client, MODEL_NAME

#Prepare function to generate content using the Gemini API
def generate_customer_reports(customer_data):

    # Fill the prompts with the customer data
    filled_zero_shot = zero_shot_template.format(**customer_data)
    filled_few_shot = few_shot_template.format(**customer_data)

    # Generate zero-shot report
    zero_shot_response = client.models.generate_content(
        model=MODEL_NAME,
        contents=filled_zero_shot
    )
    zero_shot_report = zero_shot_response.text

    # Generate few-shot report
    few_shot_response = client.models.generate_content(
        model=MODEL_NAME,
        contents=filled_few_shot
    )
    few_shot_report = few_shot_response.text

    # Return both reports
    return {
        "zero_shot_report": zero_shot_report,
        "few_shot_report": few_shot_report
    }

In [ ]:
# Example: Generate both reports for a mock customer
reports = generate_customer_reports(customer)

print("ZERO-SHOT REPORT:")
print(reports["zero_shot_report"])

print("\nFEW-SHOT REPORT:")
print(reports["few_shot_report"])

ZERO-SHOT REPORT:
Risk Summary:
The predictive model estimates an 87% churn probability (Churn Prediction: 1), indicating a very high risk of customer defection based on the provided data.

Retention Recommendations:
1. Implement a targeted cart recovery campaign. Rationale: The customer exhibits a high cart abandonment rate of 70%, suggesting potential friction in the final stages of the purchasing process.
2. Engage the customer with loyalty-building incentives. Rationale: The customer currently holds a low loyalty score of 25 and has not engaged recently (Recency Days: 45), indicating an opportunity to re-stimulate interest and program participation.

Strategic Rationale:
These recommendations directly address the customer's observed behavioral indicators—specifically high cart abandonment, low loyalty participation, and a 45-day period of inactivity—without assuming direct causation for the estimated churn risk.

FEW-SHOT REPORT:
Risk Summary:
The model estimates an 87% churn proba

### Preliminary Prompt Testing

Three synthetic customer profiles with different churn probabilities (87%, 55%, and 12%) were used to preliminarily test the Zero-Shot and Few-Shot templates.

Both templates were tested on the same customer profiles using the same Gemini model. Six reports were generated and saved in `Generative_AI/test_results.json`.




In [ ]:
# Preliminary Testing Using Mock Data

test_customers = [
    {
        "churn_prediction": 1,
        "churn_probability": "87%",
        "recency_days": 45,
        "loyalty_score": 25,
        "cart_abandonment_rate": "70%"
    },
    {
    "churn_prediction": 1,
    "churn_probability": "55%",
    "recency_days": 20,
    "loyalty_score": 55,
    "cart_abandonment_rate": "45%"
    }, 
    {
    "churn_prediction": 0,
    "churn_probability": "12%",
    "recency_days": 5,
    "loyalty_score": 90,
    "cart_abandonment_rate": "10%"
    }
]

In [ ]:
import time
from google.genai import errors

test_results = []

for i, customer in enumerate(test_customers, start=1):
    for attempt in range(3):
        try:
            reports = generate_customer_reports(customer)
            break

        except errors.ServerError:
            if attempt == 2:
                raise
            time.sleep(3)

    test_results.append({
        "test_case": f"TC{i:02d}",
        "customer_data": customer,
        "zero_shot_report": reports["zero_shot_report"],
        "few_shot_report": reports["few_shot_report"]
    })

    print(f"TC{i:02d} completed successfully!")

TC01 completed successfully!


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
import json
from pathlib import Path

output_path = Path("Generative_AI/test_results.json")

output_path.write_text(
    json.dumps(test_results, indent=4, ensure_ascii=False),
    encoding="utf-8"
)

print("Test results saved successfully!")

Test results saved successfully!


In [ ]:
saved_results = json.loads(output_path.read_text(encoding="utf-8"))

print("Saved test cases:", len(saved_results))

for result in saved_results:
    print(result["test_case"])
    print("Zero-Shot:", len(result["zero_shot_report"]), "characters")
    print("Few-Shot:", len(result["few_shot_report"]), "characters")

Saved test cases: 3
TC01
Zero-Shot: 746 characters
Few-Shot: 684 characters
TC02
Zero-Shot: 967 characters
Few-Shot: 695 characters
TC03
Zero-Shot: 664 characters
Few-Shot: 726 characters


### Qualitative Evaluation

The Zero-Shot and Few-Shot reports are evaluated to assess the quality of their generated content and their suitability for executive decision-making.

The comparison highlights the strengths and weaknesses of each template, supported by observations from the generated reports.

#### Evaluation Criteria

The generated reports are qualitatively evaluated using the following five criteria:

1. **Relevance:** Examines whether the report's risk assessment and retention recommendations are relevant to the customer's churn prediction and available information.

2. **Detail & Completeness:** Assesses whether the report provides sufficient information, includes the required recommendations, and explains the rationale behind each recommendation.

3. **Clarity & Readability:** Evaluates whether the report uses clear, understandable, and well-organized language suitable for non-technical business executives.

4. **Personalization:** Examines whether the recommendations are tailored to the customer's specific characteristics and behavioral indicators rather than providing generic advice.

5. **Safety & Factual Accuracy:** Assesses whether the report accurately represents the provided customer information, avoids unsupported assumptions or fabricated details, and does not present predictions or recommendations as guaranteed outcomes.

Each test case is analyzed by comparing the Zero-Shot and Few-Shot reports against these criteria. The observations are then summarized to identify the overall strengths and limitations of each template.

In [ ]:
result = saved_results[0]

print("Test Case:", result["test_case"])
print("\nCustomer Information:")
print(result["customer_data"])

print("\n" + "=" * 50)
print("ZERO-SHOT REPORT:")
print(result["zero_shot_report"])

print("\n" + "=" * 50)
print("FEW-SHOT REPORT:")
print(result["few_shot_report"])

Test Case: TC01

Customer Information:
{'churn_prediction': 1, 'churn_probability': '87%', 'recency_days': 45, 'loyalty_score': 25, 'cart_abandonment_rate': '70%'}

ZERO-SHOT REPORT:
Risk Summary:
The predictive model estimates an 87% probability of churn (Churn Prediction: 1), indicating a high risk of customer loss.

Retention Recommendations:
1. Deploy a targeted cart recovery campaign to address the 70% cart abandonment rate, paired with an incentive to encourage purchase completion.
2. Engage the customer through a loyalty program initiative to improve their current loyalty score of 25.

Strategic Rationale:
These recommendations directly address the customer's observed high cart abandonment rate and low loyalty score. Because information is limited to these specific behavioral features, targeted interventions in these areas are relevant ways to re-engage the customer without assuming direct causation of churn.

FEW-SHOT REPORT:
Risk Summary:
The model estimates an 87% churn proba

#### TC01 — Qualitative Comparison

| Criterion | Zero-Shot | Few-Shot |
|---|---|---|
| Relevance | Recommendations address cart abandonment and customer loyalty. | Recommendations address inactivity and cart abandonment. |
| Detail & Completeness | Provides two justified recommendations but does not discuss recency days. | Includes all provided customer indicators and two justified recommendations. |
| Clarity & Readability | Clear and structured, although some explanations are relatively lengthy. | Concise, direct, and easy to follow. |
| Personalization | Tailors recommendations to the 70% cart abandonment rate and loyalty score of 25. | Tailors recommendations to the 45 recency days and 70% cart abandonment rate. |
| Safety & Factual Accuracy | Avoids unsupported claims and explicitly acknowledges limited information. | Avoids invented facts and states that the recommendations are not guaranteed solutions. |

**Comparison Summary:**

Both templates produced relevant and actionable retention recommendations for the high-risk customer. The Few-Shot report was slightly more concise and included all provided customer indicators, while the Zero-Shot report placed greater emphasis on customer loyalty. Neither report introduced unsupported customer facts or guaranteed outcomes.

In [ ]:
result = saved_results[1]

print("Test Case:", result["test_case"])
print("\nCustomer Information:")
print(result["customer_data"])

print("\n" + "=" * 50)
print("ZERO-SHOT REPORT:")
print(result["zero_shot_report"])

print("\n" + "=" * 50)
print("FEW-SHOT REPORT:")
print(result["few_shot_report"])

Test Case: TC02

Customer Information:
{'churn_prediction': 1, 'churn_probability': '55%', 'recency_days': 20, 'loyalty_score': 55, 'cart_abandonment_rate': '45%'}

ZERO-SHOT REPORT:
Risk Summary:
The predictive model estimates a 55% probability of churn (classification: 1), indicating a moderate-to-high risk that this customer may disengage.

Retention Recommendations:
1. Targeted Cart Recovery Campaign: Send personalized reminders or incentives addressing the customer's 45% cart abandonment rate to encourage purchase completion.
2. Loyalty Program Engagement: Leverage the customer's current loyalty score of 55 by offering tailored rewards or communication to reinforce ongoing engagement within 20 days of their last activity.

Strategic Rationale:
These actions are relevant because they directly address the observed behavioral metrics—specifically the cart abandonment rate and the loyalty score—while accounting for the moderate churn probability. Since causation cannot be assumed from

#### TC02 — Qualitative Comparison

| Criterion | Zero-Shot | Few-Shot |
|---|---|---|
| Relevance | Recommendations address the customer's cart abandonment and loyalty indicators. | Recommendations also address cart abandonment and loyalty, aligning with the available customer information. |
| Detail & Completeness | Provides two recommendations with detailed explanations and incorporates all three behavioral indicators. | Includes all provided customer indicators and two recommendations with brief rationales. |
| Clarity & Readability | Clear and structured, but uses relatively lengthy explanations and more complex language. | Concise, straightforward, and easy to follow. |
| Personalization | Uses the 45% cart abandonment rate, loyalty score of 55, and 20 recency days to tailor its recommendations. | Tailors recommendations to the 45% cart abandonment rate and loyalty score of 55, while mentioning recency only in the risk summary. |
| Safety & Factual Accuracy | Avoids unsupported causal claims, but uses the 20 recency days as an engagement timeframe, although the data only indicate the time since the customer's last activity | Accurately represents the provided information and avoids unsupported guarantees or assumptions. |

**Comparison Summary:**

Both templates generated relevant and actionable recommendations for the customer. The Zero-Shot report provided more detailed explanations and incorporated recency into its recommendations, but its proposed engagement timing was not clearly supported by the available information. The Few-Shot report delivered more concise recommendations while maintaining relevance and factual accuracy. Overall, the Few-Shot report demonstrated stronger clarity, while the Zero-Shot report provided greater detail.

In [ ]:
result = saved_results[2]

print("Test Case:", result["test_case"])
print("\nCustomer Information:")
print(result["customer_data"])

print("\n" + "=" * 50)
print("ZERO-SHOT REPORT:")
print(result["zero_shot_report"])

print("\n" + "=" * 50)
print("FEW-SHOT REPORT:")
print(result["few_shot_report"])

Test Case: TC03

Customer Information:
{'churn_prediction': 0, 'churn_probability': '12%', 'recency_days': 5, 'loyalty_score': 90, 'cart_abandonment_rate': '10%'}

ZERO-SHOT REPORT:
Risk Summary:
The model estimates a low churn risk, with a predicted churn probability of 12% and a churn prediction of 0.

Retention Recommendations:
1. Maintain regular engagement through existing communication channels, supported by a recent activity timestamp of 5 days.
2. Leverage the high loyalty score of 90 by inviting the customer to an exclusive loyalty tier or rewards program.

Strategic Rationale:
These recommendations capitalize on the customer's demonstrated high engagement and strong loyalty score. Maintaining standard touchpoints and recognizing their high loyalty status are proactive measures that align with their current observed behavior.

FEW-SHOT REPORT:
Risk Summary:
The model estimates a 12% churn probability for this customer, indicating relatively low predicted churn risk. The availa

#### TC03 — Qualitative Comparison

| Criterion | Zero-Shot | Few-Shot |
|---|---|---|
| Relevance | Recommends maintaining engagement and recognizing customer loyalty, which aligns with the low predicted churn risk. | Recommends light-touch communication and loyalty appreciation, appropriately reflecting the low predicted churn risk. |
| Detail & Completeness | Provides two recommendations with rationales but does not address the 10% cart abandonment rate. | Includes all three behavioral indicators and provides two recommendations with a supporting rationale. |
| Clarity & Readability | Uses clear, structured language, although some phrases are relatively formal. | Provides concise, understandable recommendations with a clear strategic explanation. |
| Personalization | Tailors recommendations to the customer's recent activity (5 days) and high loyalty score (90). | Tailors recommendations to the customer's recent activity, high loyalty score, and low cart abandonment rate, although the connection between cart abandonment and loyalty appreciation is not clearly explained. |
| Safety & Factual Accuracy | Accurately represents the churn estimate and avoids guaranteed outcomes, but describes the customer as highly engaged based on limited indicators. | Accurately represents the provided values and avoids guaranteed outcomes, although its description of strong engagement is an interpretation of the available indicators. |

**Comparison Summary:**

Both templates generated retention recommendations appropriate for a customer with low predicted churn risk. The Zero-Shot report emphasized maintaining engagement and rewarding loyalty, while the Few-Shot report recommended a less intensive retention approach. The Few-Shot report included all available behavioral indicators and provided a clearer explanation of why aggressive intervention was unnecessary. However, its use of cart abandonment to justify loyalty appreciation was not clearly supported. Overall, both reports were relevant, with the Few-Shot report demonstrating a slightly clearer risk-based strategy.

#### Overall Qualitative Comparison

Across the three test cases, both the Zero-Shot and Few-Shot templates generated relevant retention recommendations aligned with the customers' predicted churn risks and available behavioral information.

The **Zero-Shot template** generally provided more detailed explanations and demonstrated flexibility in selecting customer indicators for its recommendations. However, its responses were sometimes more verbose, and TC02 included an unclear interpretation of recency as an engagement timeframe.

The **Few-Shot template** generally produced more concise, structured, and executive-friendly reports. It consistently included all three behavioral indicators in its risk summaries and adapted its recommendations to different churn risk levels. However, some connections between customer indicators and recommended actions were not fully justified, particularly in TC03.

Regarding **Safety & Factual Accuracy**, neither template fabricated customer values or guaranteed retention outcomes. Nevertheless, both showed limitations in how certain behavioral indicators were interpreted.

**Overall, the Few-Shot template demonstrated a slight qualitative advantage** in clarity, consistency, and completeness of customer information. The Zero-Shot template remained competitive in recommendation detail and contextual reasoning. The final template selection will also consider the quantitative evaluation results.

### Quantitative Evaluation

To objectively compare the Zero-Shot and Few-Shot prompt templates, quantitative metrics are applied to the generated reports. Both templates are evaluated using the same test cases to ensure a consistent and fair comparison.

#### 1. Response Length

Response length measures the number of words generated by each prompt template for each test case. This metric helps identify differences in the amount of information provided by the two templates.

A longer response does not necessarily indicate better quality; therefore, response length should be interpreted alongside other evaluation results.

In [ ]:
for result in saved_results:
    zero_words = len(result["zero_shot_report"].split())
    few_words = len(result["few_shot_report"].split())

    print(
        f'{result["test_case"]}: '
        f'Zero-Shot = {zero_words} words | '
        f'Few-Shot = {few_words} words'
    )

TC01: Zero-Shot = 106 words | Few-Shot = 98 words
TC02: Zero-Shot = 127 words | Few-Shot = 100 words
TC03: Zero-Shot = 95 words | Few-Shot = 103 words


In [ ]:
zero_avg = sum(len(r["zero_shot_report"].split()) for r in saved_results) / len(saved_results)
few_avg = sum(len(r["few_shot_report"].split()) for r in saved_results) / len(saved_results)

print(f"\nAverage Response Length:")
print(f"Zero-Shot: {zero_avg:.1f} words")
print(f"Few-Shot: {few_avg:.1f} words")


Average Response Length:
Zero-Shot: 109.3 words
Few-Shot: 100.3 words


The Zero-Shot template generated longer responses in two out of three test cases, with an average of 109.3 words compared to 100.3 words for the Few-Shot template.

This indicates that the Zero-Shot template produced slightly more verbose reports overall. However, the difference in response length alone does not establish which template provides more relevant, complete, or useful recommendations.

#### 2. Keyword Alignment

Keyword alignment measures the coverage of domain-specific terminology in the generated reports. Four keyword groups were selected based on the project's focus on customer churn and behavioral indicators: Customer Churn, Customer Loyalty, Customer Engagement, and Cart Abandonment.

A group is considered matched when at least one of its associated terms appears in the report. The alignment score is calculated as the percentage of matched groups out of the four groups, using the same criteria for both prompt templates.

In [ ]:
import re

keyword_groups = {
    "Customer Churn": ["churn", "attrition"],
    "Customer Loyalty": ["loyalty", "loyal"],
    "Customer Engagement": ["engagement", "re-engagement", "inactivity"],
    "Cart Abandonment": ["cart", "checkout", "abandonment"]
}

def calculate_keyword_alignment(report):
    report = report.lower()

    matched_groups = 0

    for terms in keyword_groups.values():
        if any(re.search(r"\b" + re.escape(term) + r"\b", report) for term in terms):
            matched_groups += 1

    return matched_groups / len(keyword_groups) * 100


for result in saved_results:
    zero_score = calculate_keyword_alignment(result["zero_shot_report"])
    few_score = calculate_keyword_alignment(result["few_shot_report"])

    print(
        f'{result["test_case"]}: '
        f'Zero-Shot = {zero_score:.1f}% | '
        f'Few-Shot = {few_score:.1f}%'
    )

TC01: Zero-Shot = 75.0% | Few-Shot = 100.0%
TC02: Zero-Shot = 100.0% | Few-Shot = 100.0%
TC03: Zero-Shot = 75.0% | Few-Shot = 100.0%


The Few-Shot template achieved 100% keyword alignment across all three test cases, compared to an average of 83.3% for the Zero-Shot template.

This indicates that the Few-Shot reports covered more of the selected domain terminology. However, keyword alignment measures terminology coverage rather than the relevance or quality of the recommendations, which are evaluated separately.

#### 3. Readability Score (Flesch-Kincaid)

Readability is evaluated using the Flesch-Kincaid Grade Level, which estimates the educational reading level required to understand a text based on sentence length and word complexity.

Lower scores generally indicate easier-to-read text. This metric is used to compare the accessibility of the Zero-Shot and Few-Shot reports for business executives, although it does not directly measure the accuracy or usefulness of their recommendations.

In [ ]:
%pip install textstat

Defaulting to user installation because normal site-packages is not writeable
  Obtaining dependency information for textstat from https://files.pythonhosted.org/packages/ca/31/0eb4cc5bb021b4ceaaa602c59ba16ce99256b9dd30981bef3f3a53d8555f/textstat-0.7.13-py3-none-any.whl.metadata
  Obtaining dependency information for pyphen from https://files.pythonhosted.org/packages/ec/1d/23801cf008f71575f0a4800463f349afa5f04b27fe783178a45cdc4d5edf/pyphen-0.18.1-py3-none-any.whl.metadata
  Obtaining dependency information for nltk from https://files.pythonhosted.org/packages/a2/04/e7816813e409ca874ba2fcc35455db0e4130b79eb277e1f9edd270ca3c69/nltk-3.10.4-py3-none-any.whl.metadata
  Obtaining dependency information for setuptools from https://files.pythonhosted.org/packages/ef/55/6dc6d6bb953bdfc52519a0059592c52b6e3b8ad51cf8256dffceb55b0610/setuptools-84.2.1-py3-none-any.whl.metadata
  Obtaining dependency information for defusedxml from https://files.pythonhosted.org/packages/07/6c/aa3f2f849e01cb6a001cd


[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import textstat

zero_scores = []
few_scores = []

for result in saved_results:
    zero_score = textstat.flesch_kincaid_grade(result["zero_shot_report"])
    few_score = textstat.flesch_kincaid_grade(result["few_shot_report"])

    zero_scores.append(zero_score)
    few_scores.append(few_score)

    print(
        f'{result["test_case"]}: '
        f'Zero-Shot = {zero_score:.1f} | '
        f'Few-Shot = {few_score:.1f}'
    )

print("\nAverage Readability Score:")
print(f"Zero-Shot: {sum(zero_scores) / len(zero_scores):.1f}")
print(f"Few-Shot: {sum(few_scores) / len(few_scores):.1f}")

TC01: Zero-Shot = 14.6 | Few-Shot = 13.5
TC02: Zero-Shot = 18.0 | Few-Shot = 13.1
TC03: Zero-Shot = 13.6 | Few-Shot = 15.2

Average Readability Score:
Zero-Shot: 15.4
Few-Shot: 13.9


The Few-Shot template achieved a lower average Flesch-Kincaid Grade Level (13.9) compared to the Zero-Shot template (15.4), indicating that its reports were generally easier to read according to this metric.

The Few-Shot template produced lower scores in TC01 and TC02, while the Zero-Shot template was easier to read in TC03. However, both templates generated relatively complex text, and readability scores alone do not determine whether the reports are clear or useful to business executives.

#### 4. User Preference Simulation

User preference simulation is conducted to estimate which generated report a business executive would prefer when making customer retention decisions.

For each test case, Gemini acts as a simulated business executive and compares two anonymized reports (Report A and Report B) generated for the same customer. The evaluator selects the report it would prefer to use for making a clear, practical retention decision, with the option to select a tie.

The selected preferences are recorded across the test cases, and the preference percentage for each template is calculated.

This approach simulates executive decision-making rather than collecting actual user feedback. Since Gemini is also used to generate the reports, the simulation may be affected by model bias and should not be interpreted as independent human validation.

In [ ]:
def build_preference_prompt(customer_data, report_a, report_b):
    return f"""
You are simulating a business executive responsible for customer retention decisions.

Your task is to review two AI-generated customer retention reports
for the same customer and select the report you would prefer to use
when making a retention decision.

Customer Information:
{customer_data}

Report A:
{report_a}

Report B:
{report_b}

Instructions:
- Evaluate the reports from the perspective of a business executive.
- Prefer the report that better supports a clear, practical, and
  informed customer retention decision.
- Consider whether the recommendations are appropriate for the
  customer's predicted churn risk and available information.
- Do not assume that a longer report is necessarily better.
- Do not favor a report based on its label or writing style alone.
- Select Tie if neither report has a meaningful advantage.
- Base your decision only on the provided information.

Required Output Format:
Preferred Report: A, B, or Tie
Reason: [Brief explanation of the preference]
"""

In [ ]:
result = saved_results[0]

customer_data = result["customer_data"]

# Hide the prompt template names from the evaluator
report_a = result["zero_shot_report"]
report_b = result["few_shot_report"]

preference_prompt = build_preference_prompt(
    customer_data,
    report_a,
    report_b
)

response = client.models.generate_content(
    model=MODEL_NAME,
    contents=preference_prompt
)

print("Test Case:", result["test_case"])
print("\nSimulated Executive Preference:")
print(response.text)

Test Case: TC01

Simulated Executive Preference:
Preferred Report: B

Reason: Report B incorporates all four available data points from the customer profile (87% churn probability, 45 recency days, loyalty score of 25, and 70% cart abandonment rate) into its risk summary and directly ties its recommendations to specific behavioral metrics, such as addressing the 45 days of inactivity. Report A omits the recency data and provides a more generic loyalty recommendation. Report B offers a more comprehensive and practical foundation for a retention decision.


In [ ]:
result = saved_results[1]

# Reverse the report order for TC02
report_a = result["few_shot_report"]
report_b = result["zero_shot_report"]

preference_prompt = build_preference_prompt(
    result["customer_data"],
    report_a,
    report_b
)

response = client.models.generate_content(
    model=MODEL_NAME,
    contents=preference_prompt
)

print("Test Case:", result["test_case"])
print("\nSimulated Executive Preference:")
print(response.text)

Test Case: TC02

Simulated Executive Preference:
Preferred Report: B
Reason: Report B provides more cohesive and actionable recommendations by integrating the recency metric directly into the execution timeline (noting the 20-day threshold), whereas Report A isolates the recency data without applying it. Furthermore, Report B demonstrates stronger executive framing by explicitly acknowledging the nuance of causation versus correlation, making it a more informed and reliable basis for a retention strategy.


In [ ]:
result = saved_results[2]

report_a = result["zero_shot_report"]
report_b = result["few_shot_report"]

preference_prompt = build_preference_prompt(
    result["customer_data"],
    report_a,
    report_b
)

response = client.models.generate_content(
    model=MODEL_NAME,
    contents=preference_prompt
)

print("Test Case:", result["test_case"])
print("\nSimulated Executive Preference:")
print(response.text)

Test Case: TC03

Simulated Executive Preference:
Preferred Report: A

Reason: Report A provides specific, actionable recommendations—such as inviting the customer to an exclusive loyalty tier or rewards program—which directly leverage the customer's high loyalty score of 90. Report B offers more vague suggestions ("subtle loyalty perks," "periodic appreciation message") and attempts to connect them to the 10% cart abandonment rate without clearly explaining the strategic link. Report A's recommendations are more practical and better aligned with a proactive retention strategy for a low-risk, highly loyal customer.


In [ ]:
preferences = {
    "TC01": "Few-Shot",
    "TC02": "Zero-Shot",
    "TC03": "Zero-Shot"
}

total = len(preferences)

for template in ["Zero-Shot", "Few-Shot", "Tie"]:
    count = sum(
        preference == template
        for preference in preferences.values()
    )
    percentage = count / total * 100

    print(f"{template}: {count}/{total} ({percentage:.1f}%)")

Zero-Shot: 2/3 (66.7%)
Few-Shot: 1/3 (33.3%)
Tie: 0/3 (0.0%)


**Results Interpretation**

The simulated executive preference test favored the Zero-Shot template in two out of three test cases (66.7%), while the Few-Shot template was preferred in one case (33.3%).

The Few-Shot report was preferred in TC01 because it incorporated all available customer indicators and provided practical recommendations. The Zero-Shot report was preferred in TC02 for its detailed strategic reasoning and in TC03 for its more specific loyalty recommendations.

However, the TC02 preference was partly based on an unclear interpretation of recency days as an engagement timeframe, highlighting a limitation in the simulated evaluator's judgment.

These results represent AI-simulated preferences rather than actual user feedback. The small number of test cases and the use of Gemini for both report generation and preference evaluation limit the reliability and generalizability of the findings. Therefore, the preference results should be considered alongside the qualitative analysis and other quantitative metrics when selecting the final prompt template.